# Federated Learning on Cotton Plant Disease Dataset
## Implementing FedAvg and FedProx with Non-IID Dirichlet Distribution
### Based on paper specifications: ResNet18, 5 clients, α=0.5, 10 rounds, 5 local epochs

## 1. Install Dependencies

## 2. Imports

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 3. Dataset Path & Exploration

In [ ]:
# Kaggle dataset path
DATA_DIR = '/kaggle/input/datasets/dhamur/cotton-plant-disease/Cotton leaves/40 Images'

# List available folders
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')

In [ ]:
# Auto-detect train directory
# Try common folder structures in this dataset
POSSIBLE_TRAIN_DIRS = [
    os.path.join(DATA_DIR, 'train'),
    os.path.join(DATA_DIR, 'Train'),
    os.path.join(DATA_DIR, 'training'),
    DATA_DIR
]

TRAIN_DIR = None
TEST_DIR  = None

for d in POSSIBLE_TRAIN_DIRS:
    if os.path.isdir(d):
        subdirs = [s for s in os.listdir(d) if os.path.isdir(os.path.join(d, s))]
        if len(subdirs) >= 2:
            TRAIN_DIR = d
            print(f'Train dir found: {TRAIN_DIR}')
            print(f'Classes: {subdirs}')
            break

# Try to find test dir
for possible in [os.path.join(DATA_DIR, 'test'), os.path.join(DATA_DIR, 'Test'),
                 os.path.join(DATA_DIR, 'val'), os.path.join(DATA_DIR, 'valid')]:
    if os.path.isdir(possible):
        TEST_DIR = possible
        print(f'Test/val dir found: {TEST_DIR}')
        break

if TRAIN_DIR is None:
    raise ValueError('Could not find a valid image directory. Check DATA_DIR.')

## 4. Data Transforms & Full Dataset

In [ ]:
# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load full dataset from TRAIN_DIR
full_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transforms)
NUM_CLASSES = len(full_dataset.classes)
CLASS_NAMES = full_dataset.classes

print(f'Total samples : {len(full_dataset)}')
print(f'Number of classes: {NUM_CLASSES}')
print(f'Class names  : {CLASS_NAMES}')

# Class distribution
targets = np.array(full_dataset.targets)
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c}: {(targets == i).sum()} samples')

In [ ]:
# Train / Test split (80/20) if no separate test folder
from torch.utils.data import random_split

if TEST_DIR is not None:
    train_dataset = full_dataset
    test_dataset_raw = ImageFolder(root=TEST_DIR, transform=test_transforms)
    print(f'Using separate test dir. Train: {len(train_dataset)}, Test: {len(test_dataset_raw)}')
else:
    total = len(full_dataset)
    test_size  = int(0.2 * total)
    train_size = total - test_size
    train_dataset, test_dataset_raw = random_split(
        full_dataset, [train_size, test_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    print(f'Split: Train={train_size}, Test={test_size}')

# Test dataset with test transforms
class TransformSubset(Dataset):
    """Wrap a Subset/Dataset with a different transform."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img is already a tensor if coming from ImageFolder with transforms
        # We need PIL image for re-transforming
        # Use the underlying dataset
        return img, label

if TEST_DIR is None:
    # Rebuild test subset with test_transforms using indices
    test_indices = test_dataset_raw.indices
    test_dataset_notransform = ImageFolder(root=TRAIN_DIR, transform=test_transforms)
    test_dataset = Subset(test_dataset_notransform, test_indices)

    # Rebuild train subset indices too
    train_indices = train_dataset.indices
else:
    test_dataset  = test_dataset_raw
    train_indices = list(range(len(train_dataset)))

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print('Test loader ready.')

## 5. Non-IID Dirichlet Data Distribution (α = 0.5)

In [ ]:
def dirichlet_split(dataset, indices, num_clients, num_classes, alpha, seed=42):
    """
    Partition dataset indices among clients using Dirichlet distribution.
    alpha=0.5 creates strong non-IID heterogeneity.
    Returns: list of lists of indices, one per client.
    """
    np.random.seed(seed)

    # Get labels for the given indices
    if hasattr(dataset, 'targets'):
        all_targets = np.array(dataset.targets)
    else:
        # Subset case
        all_targets = np.array(dataset.dataset.targets)

    labels = all_targets[indices]
    indices = np.array(indices)

    # Group indices by class
    class_indices = defaultdict(list)
    for idx, lbl in zip(indices, labels):
        class_indices[lbl].append(idx)

    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        class_idx = np.array(class_indices[c])
        if len(class_idx) == 0:
            continue
        np.random.shuffle(class_idx)

        # Sample proportions from Dirichlet
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        # Ensure proportions sum to 1
        proportions = proportions / proportions.sum()

        # Split class indices among clients
        splits = (proportions * len(class_idx)).astype(int)
        # Handle rounding: assign remaining to last client with samples
        diff = len(class_idx) - splits.sum()
        splits[-1] += diff

        start = 0
        for client_id, split_size in enumerate(splits):
            end = start + split_size
            client_indices[client_id].extend(class_idx[start:end].tolist())
            start = end

    return client_indices


# Configuration
NUM_CLIENTS    = 5
ALPHA          = 0.5    # Dirichlet concentration parameter
NUM_ROUNDS     = 10     # Global communication rounds
LOCAL_EPOCHS   = 5      # Local epochs per round
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
SGD_MOMENTUM   = 0.9
MU_FEDPROX     = 0.01   # FedProx proximal coefficient

# Partition training data among clients
client_indices = dirichlet_split(
    full_dataset, train_indices,
    num_clients=NUM_CLIENTS,
    num_classes=NUM_CLASSES,
    alpha=ALPHA,
    seed=SEED
)

print('Client data distribution (Non-IID, Dirichlet α=0.5):')
for i, idxs in enumerate(client_indices):
    if hasattr(full_dataset, 'targets'):
        lbls = np.array(full_dataset.targets)[idxs]
    else:
        lbls = np.array(full_dataset.dataset.targets)[idxs]
    counts = np.bincount(lbls, minlength=NUM_CLASSES)
    print(f'  Client {i+1}: {len(idxs)} samples | {dict(zip(CLASS_NAMES, counts))}')

In [ ]:
# Visualize Non-IID distribution
distribution_matrix = np.zeros((NUM_CLIENTS, NUM_CLASSES))

for i, idxs in enumerate(client_indices):
    if hasattr(full_dataset, 'targets'):
        lbls = np.array(full_dataset.targets)[idxs]
    else:
        lbls = np.array(full_dataset.dataset.targets)[idxs]
    counts = np.bincount(lbls, minlength=NUM_CLASSES)
    distribution_matrix[i] = counts

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(distribution_matrix, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES,
            yticklabels=[f'Client {i+1}' for i in range(NUM_CLIENTS)],
            ax=ax)
ax.set_title(f'Non-IID Data Distribution (Dirichlet α={ALPHA})', fontsize=14)
ax.set_xlabel('Disease Classes')
ax.set_ylabel('Clients (Farms)')
plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Architecture (ResNet18)

In [ ]:
def build_resnet18(num_classes):
    """
    ResNet18 with modified final FC layer for num_classes.
    Uses ImageNet pretrained weights.
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

# Test model build
test_model = build_resnet18(NUM_CLASSES)
print(f'ResNet18 built. Output classes: {NUM_CLASSES}')
print(f'FC layer: {test_model.fc}')
del test_model

## 7. Helper Functions

In [ ]:
def get_client_dataloader(client_id, full_dataset, client_indices, batch_size=32):
    """Create DataLoader for a specific client."""
    subset = Subset(full_dataset, client_indices[client_id])
    loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=False
    )
    return loader


def get_model_params(model):
    """Return a deep copy of model parameters as state_dict."""
    return copy.deepcopy(model.state_dict())


def set_model_params(model, params):
    """Load parameters into model."""
    model.load_state_dict(copy.deepcopy(params))


def fedavg_aggregate(global_params, client_params_list, client_sizes):
    """
    FedAvg weighted aggregation.
    Weights are proportional to number of local training samples.
    """
    total_samples = sum(client_sizes)
    aggregated = copy.deepcopy(global_params)

    for key in aggregated.keys():
        aggregated[key] = torch.zeros_like(aggregated[key], dtype=torch.float32)
        for params, size in zip(client_params_list, client_sizes):
            aggregated[key] += params[key].float() * (size / total_samples)

    return aggregated


def evaluate_model(model, data_loader, device):
    """
    Evaluate model on a DataLoader.
    Returns: accuracy, precision, recall, f1, all_preds, all_labels
    """
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    return acc, prec, rec, f1, all_preds, all_labels


print('Helper functions defined.')

## 8. FedAvg Implementation

In [ ]:
def local_train_fedavg(model, data_loader, local_epochs, lr, momentum, device):
    """
    Local training for FedAvg.
    Optimizer: SGD, momentum=0.9, lr=0.001
    """
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_batches = 0

    for epoch in range(local_epochs):
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss   += loss.item()
            total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    return avg_loss


def run_fedavg(num_rounds, num_clients, local_epochs, lr, momentum,
               batch_size, full_dataset, client_indices, num_classes,
               test_loader, device):
    """
    Full FedAvg training loop.
    - All clients participate every round
    - Weighted aggregation by local dataset size
    """
    print('\n' + '='*60)
    print('         FEDAVG TRAINING')
    print('='*60)

    # Initialize global model
    global_model = build_resnet18(num_classes).to(device)
    global_params = get_model_params(global_model)

    history = {
        'round': [], 'accuracy': [], 'precision': [],
        'recall': [], 'f1': [], 'avg_client_loss': []
    }

    client_sizes = [len(client_indices[c]) for c in range(num_clients)]

    for rnd in range(1, num_rounds + 1):
        print(f'\nRound {rnd}/{num_rounds}')
        client_params_list = []
        round_losses = []

        for client_id in range(num_clients):
            # Each client gets a fresh copy of global model
            local_model = build_resnet18(num_classes).to(device)
            set_model_params(local_model, global_params)

            # Get client dataloader
            loader = get_client_dataloader(client_id, full_dataset, client_indices, batch_size)

            # Local training
            loss = local_train_fedavg(local_model, loader, local_epochs, lr, momentum, device)
            round_losses.append(loss)

            client_params_list.append(get_model_params(local_model))
            del local_model
            torch.cuda.empty_cache()

            print(f'  Client {client_id+1} | Loss: {loss:.4f} | Samples: {client_sizes[client_id]}')

        # Aggregate: FedAvg weighted average
        global_params = fedavg_aggregate(global_params, client_params_list, client_sizes)
        set_model_params(global_model, global_params)

        # Evaluate global model on test set
        acc, prec, rec, f1, _, _ = evaluate_model(global_model, test_loader, device)
        avg_loss = np.mean(round_losses)

        history['round'].append(rnd)
        history['accuracy'].append(acc)
        history['precision'].append(prec)
        history['recall'].append(rec)
        history['f1'].append(f1)
        history['avg_client_loss'].append(avg_loss)

        print(f'  >> Global  | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Avg Loss: {avg_loss:.4f}')

    print('\nFedAvg Training Complete!')
    return global_model, history


print('FedAvg functions defined.')

## 9. FedProx Implementation

In [ ]:
def local_train_fedprox(model, global_params, data_loader, local_epochs,
                        lr, momentum, mu, device):
    """
    Local training for FedProx.
    Adds proximal regularization: (mu/2) * ||w - w_global||^2
    mu = 0.01 as per paper specification.
    """
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Store global params as tensors on device for proximal term
    global_tensors = {k: v.to(device).float() for k, v in global_params.items()}

    total_loss = 0.0
    total_batches = 0

    for epoch in range(local_epochs):
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            ce_loss = criterion(outputs, labels)

            # Proximal term: (mu/2) * ||w - w_global||^2
            prox_term = torch.tensor(0.0, device=device, requires_grad=False)
            for name, param in model.named_parameters():
                if name in global_tensors:
                    diff = param.float() - global_tensors[name]
                    prox_term = prox_term + torch.norm(diff) ** 2

            loss = ce_loss + (mu / 2.0) * prox_term
            loss.backward()
            optimizer.step()

            total_loss   += ce_loss.item()   # Track only CE loss for comparability
            total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    return avg_loss


def run_fedprox(num_rounds, num_clients, local_epochs, lr, momentum, mu,
                batch_size, full_dataset, client_indices, num_classes,
                test_loader, device):
    """
    Full FedProx training loop.
    - All clients participate every round
    - Proximal term penalizes deviation from global model
    - Aggregation same as FedAvg (weighted average)
    """
    print('\n' + '='*60)
    print('         FEDPROX TRAINING  (μ = {})'  .format(mu))
    print('='*60)

    # Initialize global model
    global_model = build_resnet18(num_classes).to(device)
    global_params = get_model_params(global_model)

    history = {
        'round': [], 'accuracy': [], 'precision': [],
        'recall': [], 'f1': [], 'avg_client_loss': []
    }

    client_sizes = [len(client_indices[c]) for c in range(num_clients)]

    for rnd in range(1, num_rounds + 1):
        print(f'\nRound {rnd}/{num_rounds}')
        client_params_list = []
        round_losses = []

        for client_id in range(num_clients):
            # Each client gets a fresh copy of global model
            local_model = build_resnet18(num_classes).to(device)
            set_model_params(local_model, global_params)

            # Get client dataloader
            loader = get_client_dataloader(client_id, full_dataset, client_indices, batch_size)

            # Local training with proximal term
            loss = local_train_fedprox(
                local_model, global_params, loader,
                local_epochs, lr, momentum, mu, device
            )
            round_losses.append(loss)

            client_params_list.append(get_model_params(local_model))
            del local_model
            torch.cuda.empty_cache()

            print(f'  Client {client_id+1} | CE Loss: {loss:.4f} | Samples: {client_sizes[client_id]}')

        # Aggregate: same weighted FedAvg aggregation
        global_params = fedavg_aggregate(global_params, client_params_list, client_sizes)
        set_model_params(global_model, global_params)

        # Evaluate global model on test set
        acc, prec, rec, f1, _, _ = evaluate_model(global_model, test_loader, device)
        avg_loss = np.mean(round_losses)

        history['round'].append(rnd)
        history['accuracy'].append(acc)
        history['precision'].append(prec)
        history['recall'].append(rec)
        history['f1'].append(f1)
        history['avg_client_loss'].append(avg_loss)

        print(f'  >> Global  | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Avg Loss: {avg_loss:.4f}')

    print('\nFedProx Training Complete!')
    return global_model, history


print('FedProx functions defined.')

## 10. Run FedAvg

In [ ]:
fedavg_model, fedavg_history = run_fedavg(
    num_rounds    = NUM_ROUNDS,
    num_clients   = NUM_CLIENTS,
    local_epochs  = LOCAL_EPOCHS,
    lr            = LEARNING_RATE,
    momentum      = SGD_MOMENTUM,
    batch_size    = BATCH_SIZE,
    full_dataset  = full_dataset,
    client_indices= client_indices,
    num_classes   = NUM_CLASSES,
    test_loader   = test_loader,
    device        = DEVICE
)

## 11. Run FedProx

In [ ]:
fedprox_model, fedprox_history = run_fedprox(
    num_rounds    = NUM_ROUNDS,
    num_clients   = NUM_CLIENTS,
    local_epochs  = LOCAL_EPOCHS,
    lr            = LEARNING_RATE,
    momentum      = SGD_MOMENTUM,
    mu            = MU_FEDPROX,
    batch_size    = BATCH_SIZE,
    full_dataset  = full_dataset,
    client_indices= client_indices,
    num_classes   = NUM_CLASSES,
    test_loader   = test_loader,
    device        = DEVICE
)

## 12. Final Evaluation & Metrics

In [ ]:
print('\n' + '='*60)
print('         FINAL TEST SET EVALUATION')
print('='*60)

# FedAvg final
fa_acc, fa_prec, fa_rec, fa_f1, fa_preds, fa_labels = evaluate_model(fedavg_model, test_loader, DEVICE)
print(f'\nFedAvg  | Acc: {fa_acc:.4f} | Prec: {fa_prec:.4f} | Rec: {fa_rec:.4f} | F1: {fa_f1:.4f}')

# FedProx final
fp_acc, fp_prec, fp_rec, fp_f1, fp_preds, fp_labels = evaluate_model(fedprox_model, test_loader, DEVICE)
print(f'FedProx | Acc: {fp_acc:.4f} | Prec: {fp_prec:.4f} | Rec: {fp_rec:.4f} | F1: {fp_f1:.4f}')

# Summary table
results_df = pd.DataFrame({
    'Algorithm' : ['FedAvg', 'FedProx'],
    'Accuracy'  : [fa_acc,   fp_acc],
    'Precision' : [fa_prec,  fp_prec],
    'Recall'    : [fa_rec,   fp_rec],
    'F1-Score'  : [fa_f1,    fp_f1],
})
results_df = results_df.round(4)
print('\nSummary Table:')
print(results_df.to_string(index=False))
results_df.to_csv('federated_results_summary.csv', index=False)
print('\nSaved: federated_results_summary.csv')

## 13. Classification Reports

In [ ]:
print('\nFedAvg Classification Report:')
print(classification_report(fa_labels, fa_preds, target_names=CLASS_NAMES, zero_division=0))

print('\nFedProx Classification Report:')
print(classification_report(fp_labels, fp_preds, target_names=CLASS_NAMES, zero_division=0))

## 14. Visualization

In [ ]:
# ── Plot 1: Accuracy & F1 over rounds ──────────────────────────────────────
rounds = list(range(1, NUM_ROUNDS + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FedAvg vs FedProx – Training Metrics over Communication Rounds', fontsize=14)

metrics = [('accuracy', 'Accuracy'), ('precision', 'Precision'),
           ('recall',   'Recall'),   ('f1',         'F1-Score')]

for ax, (key, title) in zip(axes.flatten(), metrics):
    ax.plot(rounds, fedavg_history[key],  'b-o', label='FedAvg',  linewidth=2, markersize=5)
    ax.plot(rounds, fedprox_history[key], 'r-s', label='FedProx', linewidth=2, markersize=5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Communication Round')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xticks(rounds)

plt.tight_layout()
plt.savefig('metrics_over_rounds.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: Average client loss over rounds ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rounds, fedavg_history['avg_client_loss'],  'b-o', label='FedAvg',  linewidth=2)
ax.plot(rounds, fedprox_history['avg_client_loss'], 'r-s', label='FedProx', linewidth=2)
ax.set_title('Average Client Training Loss per Round', fontsize=13)
ax.set_xlabel('Communication Round')
ax.set_ylabel('Avg Cross-Entropy Loss')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_xticks(rounds)
plt.tight_layout()
plt.savefig('client_loss_over_rounds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Confusion Matrices ───────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(max(6, len(class_names)*1.4), max(5, len(class_names)*1.2)))
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

plot_confusion_matrix(fa_labels, fa_preds, CLASS_NAMES,
                      'FedAvg – Confusion Matrix', 'cm_fedavg.png')

plot_confusion_matrix(fp_labels, fp_preds, CLASS_NAMES,
                      'FedProx – Confusion Matrix', 'cm_fedprox.png')

In [ ]:
# ── Plot 4: Final Metric Bar Chart ───────────────────────────────────────────
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
fedavg_vals  = [fa_acc, fa_prec, fa_rec, fa_f1]
fedprox_vals = [fp_acc, fp_prec, fp_rec, fp_f1]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, fedavg_vals,  width, label='FedAvg',  color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, fedprox_vals, width, label='FedProx', color='tomato',    edgecolor='black')

ax.set_ylim(0, 1.12)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Final Evaluation Metrics – FedAvg vs FedProx', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('final_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Save Round-wise Metrics to CSV

In [ ]:
fedavg_df  = pd.DataFrame(fedavg_history)
fedprox_df = pd.DataFrame(fedprox_history)

fedavg_df['algorithm']  = 'FedAvg'
fedprox_df['algorithm'] = 'FedProx'

all_results = pd.concat([fedavg_df, fedprox_df], ignore_index=True)
all_results = all_results.round(4)
all_results.to_csv('federated_round_metrics.csv', index=False)

print('Round-wise metrics saved to: federated_round_metrics.csv')
print(all_results.to_string(index=False))

## 16. Save Models

In [ ]:
torch.save(fedavg_model.state_dict(),  'fedavg_global_model.pth')
torch.save(fedprox_model.state_dict(), 'fedprox_global_model.pth')
print('Models saved: fedavg_global_model.pth, fedprox_global_model.pth')
print('\nAll done! ✓')